# File Operations

Goal: browse the different kinds of file operations in Python — writing, reading (whole file, line-by-line, all lines), appending, `r+`/`x` modes, checking/deleting files, `pathlib`, CSV, JSON, binary files, and exception handling around file I/O.

All examples read/write real files under `static/` (created alongside this notebook) so the outputs below reflect what actually happened on disk.

## 1. Writing to a file (`'w'` mode)

`'w'` opens a file for writing, creating it if missing and truncating it if it exists. `write()` does not add newlines automatically — you write them explicitly. The `with` statement ensures the file is closed automatically, even if an error occurs.

In [15]:
with open("static/sample.txt", "w") as f:
    f.write("Hello, file!\n")
    f.write("This is line two.\n")  # write() does not add newlines automatically

print("written")

written


## 2. Reading a whole file (`read()`)

`'r'` opens a file for reading (the default mode). `read()` returns the entire contents as a single string.

In [16]:
with open("static/sample.txt", "r") as f:
    content = f.read()  # reads the entire file as one string

print(content)
print(repr(content))

Hello, file!
This is line two.

'Hello, file!\nThis is line two.\n'


## 3. Reading line by line (`readline()`)

`readline()` reads and returns one line at a time, including its trailing `\n`. It returns an empty string once the end of the file is reached.

In [17]:
with open("static/sample.txt", "r") as f:
    first_line = f.readline()  # reads a single line, including the trailing "\n"
    second_line = f.readline()
    third_line = f.readline()  # empty string once past the end of the file

print(repr(first_line))
print(repr(second_line))
print(repr(third_line))

'Hello, file!\n'
'This is line two.\n'
''


## 4. Reading all lines (`readlines()`) and direct iteration

`readlines()` returns every line as a list. A file object is itself iterable, so looping over it directly with `for line in f` is the idiomatic way to process a file line by line without loading it all as a list.

In [18]:
with open("static/sample.txt", "r") as f:
    lines = f.readlines()  # reads all lines into a list, each keeping its "\n"

print(lines)

with open("static/sample.txt", "r") as f:
    for line in f:  # iterating the file object directly, one line at a time
        print(line.strip())

['Hello, file!\n', 'This is line two.\n']
Hello, file!
This is line two.


## 5. Appending (`'a'` mode)

`'a'` opens a file for writing without truncating it — new content is added after whatever is already there.

In [5]:
with open("static/sample.txt", "a") as f:
    f.write("Appended line.\n")  # 'a' mode adds to the end without erasing existing content

with open("static/sample.txt", "r") as f:
    print(f.read())

Hello, file!
This is line two.
Appended line.



## 6. Writing multiple lines (`writelines()`)

`writelines()` writes an iterable of strings one after another — unlike `print`, it does not insert separators or newlines for you.

In [6]:
lines_to_write = ["one\n", "two\n", "three\n"]

with open("static/numbers.txt", "w") as f:
    f.writelines(lines_to_write)  # writes a list of strings without adding separators

with open("static/numbers.txt", "r") as f:
    print(f.read())

one
two
three



## 7. Read and write together (`'r+'` mode)

`'r+'` opens an existing file for both reading and writing without truncating it. The file position after a `read()` is at the end, so a subsequent `write()` appends there.

In [7]:
with open("static/numbers.txt", "r+") as f:  # 'r+': read and write, no truncation
    content = f.read()
    f.write("four\n")  # appended right where the read left off (end of file)

with open("static/numbers.txt", "r") as f:
    print(f.read())

one
two
three
four



## 8. Exclusive creation (`'x'` mode)

`'x'` creates a new file and fails with `FileExistsError` if it already exists — useful when you specifically want to avoid overwriting something.

In [8]:
import os

if os.path.exists("static/exclusive.txt"):
    os.remove("static/exclusive.txt")  # start from a clean slate for the demo

with open("static/exclusive.txt", "x") as f:  # 'x': create, fail if it already exists
    f.write("created exclusively\n")

try:
    with open("static/exclusive.txt", "x") as f:
        f.write("won't get here\n")
except FileExistsError as e:
    print(f"caught: {e}")

caught: [Errno 17] File exists: 'static/exclusive.txt'


## 9. Checking files with `os.path` / `os`

Common inspection helpers: whether a path exists, its size, whether it's a file or directory, listing a directory's contents, and removing a file.

In [9]:
import os

print(os.path.exists("static/sample.txt"))
print(os.path.getsize("static/sample.txt"), "bytes")
print(os.path.isfile("static/sample.txt"))
print(os.path.isdir("static"))
print(sorted(os.listdir("static")))

os.remove("static/exclusive.txt")  # cleanup
print(os.path.exists("static/exclusive.txt"))

True
46 bytes
True
True
['binary_demo.bin', 'data.json', 'exclusive.txt', 'numbers.txt', 'people.csv', 'sample.txt']
False


## 10. `pathlib` — an object-oriented alternative

`pathlib.Path` wraps file operations as methods on a path object — `write_text`/`read_text` skip the explicit `open()`/`close()` dance, and path components (`suffix`, `stem`, `name`) are attributes instead of string parsing.

In [10]:
from pathlib import Path

path = Path("static/pathlib_demo.txt")
path.write_text("written via pathlib\n")  # no explicit open()/close() needed

print(path.exists())
print(path.read_text())
print(path.suffix, path.stem, path.name)

path.unlink()  # delete the file
print(path.exists())

True
written via pathlib

.txt pathlib_demo pathlib_demo.txt
False


## 11. CSV files

The `csv` module reads/writes rows as lists or (with `DictWriter`/`DictReader`) as dictionaries keyed by column header. `newline=""` avoids extra blank lines on some platforms.

In [11]:
import csv

rows = [
    {"name": "Alice", "age": 30},
    {"name": "Bob", "age": 25},
]

with open("static/people.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age"])
    writer.writeheader()
    writer.writerows(rows)

with open("static/people.csv", "r", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

{'name': 'Alice', 'age': '30'}
{'name': 'Bob', 'age': '25'}


## 12. JSON files

`json.dump`/`json.load` serialize Python objects (dicts, lists, etc.) to and from a file directly, without manually handling strings.

In [20]:
import json

data = {"name": "Alice", "age": 30, "tags": ["admin", "user"]}

with open("static/data.json", "w") as f:
    json.dump(data, f, indent=4)  # serialize a dict/list straight to a file

with open("static/data.json", "r") as f:
    loaded = json.load(f)  # parse JSON back into Python objects

print(loaded)
print(loaded["tags"])
print(loaded == data)

{'name': 'Alice', 'age': 30, 'tags': ['admin', 'user']}
['admin', 'user']
True


## 13. Binary files

`'wb'`/`'rb'` read and write raw `bytes` instead of decoded text — needed for non-text data like images, or any time text decoding should be avoided.

In [13]:
data = bytes([72, 101, 108, 108, 111])  # "Hello" as raw bytes

with open("static/binary_demo.bin", "wb") as f:  # 'wb': write binary
    f.write(data)

with open("static/binary_demo.bin", "rb") as f:  # 'rb': read binary
    raw = f.read()

print(raw)
print(raw.decode("utf-8"))

b'Hello'
Hello


## 14. Exception handling around file I/O

Opening a missing file raises `FileNotFoundError`. `with` guarantees the file gets closed even if an exception is raised inside the block — no `finally: f.close()` needed.

In [14]:
try:
    with open("static/does_not_exist.txt", "r") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"caught: {e}")

# with automatically closes the file even if an error happens inside the block
try:
    with open("static/sample.txt", "r") as f:
        raise ValueError("something went wrong mid-read")
except ValueError as e:
    print(f"caught: {e}")
    print("file closed:", f.closed)

caught: [Errno 2] No such file or directory: 'static/does_not_exist.txt'
caught: something went wrong mid-read
file closed: True
